# PCBSegClassNet — Colab Training

Train PCBSegNet (segmentation) and PCBClassNet (classification) on Google Colab GPU.

**Why Colab?** Local 8 GB GPU (e.g. RTX 4060 Ti) is too tight for `batch=16` at 512×512 input — decoder activation alone is ~4 GB. Colab T4 (16 GB) or A100 (40 GB) handles it comfortably.

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 / A100 / L4 — whatever you have).
2. Have your dataset ready as a zip in Google Drive (see *Data layout* below).
3. Mount Drive when prompted in the relevant cell.

## 1. GPU sanity check

In [ ]:
!nvidia-smi

## 2. Clone the repo

If you forked it, change the URL to your fork.

In [ ]:
%cd /content
!rm -rf PCBSegClassNet
!git clone -b colab https://github.com/ironmanizawesome/PCBSegClassNet.git
%cd PCBSegClassNet

## 3. Install dependencies

Pin to TF 2.10 (the version this repo was authored against). Colab's bundled TF is often newer (Keras 3) which breaks `tf.keras.activations.softmax(...)` patterns and a few other APIs in this codebase.

In [ ]:
!pip install -q \
    tensorflow==2.10.1 \
    keras==2.10.0 \
    tensorflow-estimator==2.10.0 \
    protobuf==3.19.6 \
    numpy==1.24.4

!pip install -q albumentations==1.4.18 opencv-python-headless pyyaml tqdm pandas scikit-learn

In [ ]:
import tensorflow as tf
print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 4. Mount Drive and unpack dataset

### Data layout expected on Drive
Recommended: zip the prepared dataset and store it on Drive.

```
/MyDrive/PCBSegClassNet/
    data.zip                      ← zipped contents of data/  (segmentation/, classification/)
    checkpoints/                  ← (optional, for resume / saved best models)
```

Inside `data.zip` the structure should match what `create_patches.py` produced:
```
segmentation/train/images/*.png
segmentation/train/masks/*.png
segmentation/val/images/*.png
segmentation/val/masks/*.png
classification/train/<CLASS>/*.png
classification/val/<CLASS>/*.png
```

Why unzip to local disk and not stream from Drive? Drive mounts thousands of small files extremely slowly (API throttling). Always unpack to `/content` for training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Adjust path if you stored data.zip elsewhere
DATA_ZIP = "/content/drive/MyDrive/PCBSegClassNet/data.zip"

import os, time
assert os.path.exists(DATA_ZIP), f"Not found: {DATA_ZIP}"

%cd /content/PCBSegClassNet
!mkdir -p data
t0 = time.time()
!unzip -q -o {DATA_ZIP} -d data/
print(f"Unzip done in {time.time()-t0:.1f}s")

!echo "--- segmentation ---"; ls data/segmentation/ 2>/dev/null
!echo "--- classification ---"; ls data/classification/ 2>/dev/null

## 5. (Optional) Mirror checkpoints to Drive for persistence

Colab local disk is wiped on session end. Save best model files back to Drive at the end of training (or set up a callback). For now, just record the path.

In [ ]:
DRIVE_CKPT_DIR = "/content/drive/MyDrive/PCBSegClassNet/checkpoints"
!mkdir -p {DRIVE_CKPT_DIR}

## 6. Train segmentation

Default config in `cfs/pscn_seg.yml` is `batch_size=16`, `epochs` controlled by `-epoch`.

**First run a 5-epoch sanity pass.** If loss is finite and val_dice_coef is improving, kick off the full 100 epochs.

In [ ]:
%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 5

In [ ]:
# Full training run
%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 100

In [ ]:
# Backup the best seg checkpoint to Drive
!cp /content/PCBSegClassNet/checkpoints/best_seg.h5 {DRIVE_CKPT_DIR}/best_seg.h5
!ls -la {DRIVE_CKPT_DIR}

## 7. Train classification

In [ ]:
%cd /content/PCBSegClassNet/src
!python train_classification.py -opt cfs/pscn_class.yml -epoch 5

In [ ]:
%cd /content/PCBSegClassNet/src
!python train_classification.py -opt cfs/pscn_class.yml -epoch 100

In [ ]:
# Backup the best classification checkpoint to Drive
!cp /content/PCBSegClassNet/checkpoints/best_class.h5 {DRIVE_CKPT_DIR}/best_class.h5
!ls -la {DRIVE_CKPT_DIR}

## 8. (Optional) Evaluate without retraining

Pass `-epoch 0` to skip training; the script will load `best_*.h5` from `checkpoints/` and run `model.evaluate(val_dataset)`. Make sure the checkpoint is in `/content/PCBSegClassNet/checkpoints/` (copy it back from Drive if you reconnected).

In [ ]:
# Restore checkpoints from Drive after a fresh session
!mkdir -p /content/PCBSegClassNet/checkpoints
!cp {DRIVE_CKPT_DIR}/best_seg.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no seg ckpt'
!cp {DRIVE_CKPT_DIR}/best_class.h5 /content/PCBSegClassNet/checkpoints/ 2>/dev/null || echo 'no class ckpt'

%cd /content/PCBSegClassNet/src
!python train_segmentation.py -opt cfs/pscn_seg.yml -epoch 0